In [1]:
##this notebook is to spoof a target sequence using atomworks/modelhub and outputs two files --> a .cif and a .json file for rfd3

In [5]:

##print a json file for rfd3 input
name = "test"
seq = 'AGSDRE(M3L)PLDEG'
binder_length = 100

##for the length calculations treat anything in brackets as a single residue
import re
cleaned_seq = re.sub(r'\(.*?\)', 'X', seq)
seq_count = len(cleaned_seq)


sequence_length = len(cleaned_seq)
total_length = binder_length + sequence_length
dir = './'

print(f"Name: {name}")
print(f"Sequence: {seq}")
print(f"Sequence length: {sequence_length}")
print(f"Binder length: {binder_length}")
print(f"Total length: {total_length}")

Name: testest
Sequence: AGSDRE(M3L)PLDEG
Sequence length: 12
Binder length: 100
Total length: 112


In [6]:
import json
import pickle
from os import PathLike
from pathlib import Path

from cifutils.tools.inference import (
    build_msa_paths_by_chain_id_from_component_list,
    components_to_atom_array,
)
from cifutils.utils.io_utils import to_cif_file



def _spoof_cif_from_dictionary(item: dict, temp_dir: PathLike) -> Path:
    """Unpacks a dictionary to create a CIF file from its components.

    Args:
        item (dict): A dictionary containing 'name' and 'components', optionally 'bonds'.
        temp_dir (Path): Path to the temporary directory for storing CIF files.

    Returns:
        Path: The path to the created CIF file, saved in the temporary directory.

    Raises:
        NotImplementedError: If 'bonds' is present in the dictionary.
        ValueError: If 'name' or 'components' are missing from the dictionary.
    """
    # Validate the dictionary structure ("name" and "components" are required, "bonds" is optional)
    assert (
        "name" in item and "components" in item
    ), "The input dictionary must contain 'name' and 'components' keys."

    # Build components
    atom_array, component_list = components_to_atom_array(
        item["components"], return_components=True, bonds=item.get("bonds", None)
    )
    msa_paths_by_chain_id = build_msa_paths_by_chain_id_from_component_list(
        component_list
    )

    # Create a temporary CIF file from the JSON data
    cif_path = Path(temp_dir) / f"{item['name']}.cif"
    save_path = to_cif_file(
        atom_array,
        cif_path,
        extra_categories={"msa_paths_by_chain_id": msa_paths_by_chain_id}
        if msa_paths_by_chain_id
        else None,
        file_type="cif",  # Not zipped for efficiency (as it's a temporary directory anyways)
    )

    return Path(save_path)

##need to print out the different types of components and how it is formatted in the array

    print(f"CIF file created at: {cif_file_path}")

In [7]:

# Call the function with the defined item and temp_dir
spoof_cif_path = _spoof_cif_from_dictionary(
    item={
        "name": name,
        "components": [
            {
                "seq": seq,
                "chain_id": "B"
            }
        ],
    },
    temp_dir=dir,
)

# You can then print the path to the created file
print(f"Spoofed CIF file created at: {spoof_cif_path}")


Spoofed CIF file created at: /net/scratch/magnusb/git/foundry/examples/testest.cif


In [9]:
##for the spoof file replace all the 'nan' with 0

spoofed_cif_path_str = str(spoof_cif_path)
with open(spoofed_cif_path_str, 'r') as file:
    cif_data = file.read()
cif_data = cif_data.replace('nan', '0')
with open(spoofed_cif_path_str, 'w') as file:
    file.write(cif_data)

In [10]:
# ##example json for rfd3 input 

# {
#     "cd3e": {
#         "input": "/net/scratch/magnusb/git/aa_design_ptm_pr_clean/projects/aa_design/benchmarks/1A81.cif",
#         "contig": "100-100,/0,L1-13",
#         "contig_atoms": "{}",
#         "length": "113-113",
#         "redesign_motif_sidechains": false,
#         "unfix_all": true
#     }
# }

In [11]:


rfd3_input = {
    name: {
        "input": str(spoof_cif_path),
        "contig": f"{binder_length}-{binder_length},/0,B1-{sequence_length}",
        "contig_atoms": "{}",
        "length": f"{total_length}-{total_length}",
        "redesign_motif_sidechains": False,
        "unfix_all": True
    }
}

In [12]:
print(rfd3_input)

##save as json file
with open(f"{dir}/{name}.json", "w") as json_file:
    json.dump(rfd3_input, json_file, indent=4)



{'testest': {'input': '/net/scratch/magnusb/git/foundry/examples/testest.cif', 'contig': '100-100,/0,B1-12', 'contig_atoms': '{}', 'length': '112-112', 'redesign_motif_sidechains': False, 'unfix_all': True}}
